# Face Extraction from Video using YOLOv11

This notebook uses a pre-trained YOLOv11 face detection model to extract faces from videos.

## What this notebook does:
1. Loads a pre-trained YOLOv11 model
2. Processes a video frame by frame
3. Detects faces in each frame
4. Extracts and saves cropped face images
5. Creates an annotated video with bounding boxes
6. Provides statistics on detected faces

# STEP 1: Install Required Libraries

In [1]:
# Install ultralytics (YOLOv11)
!pip install ultralytics --quiet

print("✓ Libraries installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.1 MB/s eta 0:00:00
✓ Libraries installed successfully!


# STEP 2: Import Libraries

In [2]:
import cv2
import os
from pathlib import Path
import numpy as np
from PIL import Image
from ultralytics import YOLO
import glob
from IPython.display import display

print("✓ All libraries imported successfully!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✓ All libraries imported successfully!


# STEP 3: Configure Paths

**Important:** Update these paths before running!

- `MODEL_PATH`: Path to your trained model weights file (best.pt)
- `VIDEO_PATH`: Path to the video you want to process
- `OUTPUT_DIR`: Where to save the extracted faces and annotated video

In [3]:
# ==================== UPDATE THESE PATHS ====================

# Path to your trained model weights
# For Kaggle: '/kaggle/input/your-model-dataset/best.pt'
# For local: '/path/to/your/best.pt'
MODEL_PATH = '/kaggle/input/yolo-face-model/best.pt'

# Path to your input video
# For Kaggle: '/kaggle/input/your-video-dataset/video.mp4'
# For local: '/path/to/your/video.mp4'
VIDEO_PATH = '/kaggle/input/input-video/your_video.mp4'

# Output directory for results
OUTPUT_DIR = '/kaggle/working/face_extraction_results'

# ============================================================

print(f"Model Path: {MODEL_PATH}")
print(f"Video Path: {VIDEO_PATH}")
print(f"Output Directory: {OUTPUT_DIR}")

Model Path: /kaggle/input/yolo-face-model/best.pt
Video Path: /kaggle/input/input-video/your_video.mp4
Output Directory: /kaggle/working/face_extraction_results


# STEP 4: Load Pre-trained Model

In [4]:
# Load the trained YOLOv11 face detection model
print("Loading model...")
model = YOLO(MODEL_PATH)

print("\n✓ Model loaded successfully!")
print(f"Model: {MODEL_PATH}")

Loading model...


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/yolo-face-model/best.pt'

# STEP 5: Define Video Processing Function

In [ ]:
def extract_faces_from_video(
    video_path,
    model,
    output_dir='face_extraction_results',
    frame_skip=1,
    conf_threshold=0.25,
    save_annotated_video=True,
    save_cropped_faces=True,
    save_frames=False
):
    """
    Extract faces from a video using a trained YOLO model.
    
    Parameters:
    -----------
    video_path : str
        Path to the input video file
    model : YOLO
        Trained YOLO model for face detection
    output_dir : str
        Directory to save outputs
    frame_skip : int
        Process every Nth frame (1 = all frames, 2 = every other frame, etc.)
    conf_threshold : float
        Confidence threshold for detections (0-1)
    save_annotated_video : bool
        Save video with bounding boxes drawn
    save_cropped_faces : bool
        Save individual cropped face images
    save_frames : bool
        Save all processed frames
    
    Returns:
    --------
    dict : Statistics about the processing
    """
    
    # Create output directories
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    if save_cropped_faces:
        faces_dir = output_path / 'cropped_faces'
        faces_dir.mkdir(exist_ok=True)
    
    if save_frames:
        frames_dir = output_path / 'frames'
        frames_dir.mkdir(exist_ok=True)
    
    # Open video
    cap = cv2.VideoCapture(str(video_path))
    
    if not cap.isOpened():
        raise ValueError(f"Could not open video file: {video_path}")
    
    # Get video properties
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    print(f"\n{'='*60}")
    print(f"Video Information:")
    print(f"  Resolution: {width}x{height}")
    print(f"  FPS: {fps}")
    print(f"  Total Frames: {total_frames}")
    print(f"  Duration: {total_frames/fps:.2f} seconds")
    print(f"{'='*60}\n")
    
    # Prepare video writer if saving annotated video
    if save_annotated_video:
        output_video_path = output_path / 'annotated_video.mp4'
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out_video = cv2.VideoWriter(
            str(output_video_path), 
            fourcc, 
            fps // frame_skip,
            (width, height)
        )
    
    # Statistics
    stats = {
        'total_frames_processed': 0,
        'total_faces_detected': 0,
        'frames_with_faces': 0,
        'face_count_per_frame': []
    }
    
    frame_count = 0
    face_id = 0
    
    print("Processing video...")
    
    while True:
        ret, frame = cap.read()
        
        if not ret:
            break
        
        # Skip frames if needed
        if frame_count % frame_skip != 0:
            frame_count += 1
            continue
        
        # Run face detection
        results = model.predict(
            frame, 
            conf=conf_threshold,
            verbose=False
        )
        
        # Process results
        num_faces = 0
        annotated_frame = frame.copy()
        
        for result in results:
            boxes = result.boxes
            
            for box in boxes:
                # Get bounding box coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
                conf = box.conf[0].cpu().numpy()
                
                # Draw bounding box on frame
                cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(
                    annotated_frame, 
                    f'Face {conf:.2f}', 
                    (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 
                    0.5, 
                    (0, 255, 0), 
                    2
                )
                
                # Crop and save face
                if save_cropped_faces:
                    face_crop = frame[y1:y2, x1:x2]
                    if face_crop.size > 0:
                        face_path = faces_dir / f'face_{frame_count:06d}_{face_id:04d}.jpg'
                        cv2.imwrite(str(face_path), face_crop)
                        face_id += 1
                
                num_faces += 1
        
        # Update statistics
        stats['total_frames_processed'] += 1
        stats['total_faces_detected'] += num_faces
        stats['face_count_per_frame'].append(num_faces)
        if num_faces > 0:
            stats['frames_with_faces'] += 1
        
        # Save annotated frame to video
        if save_annotated_video:
            out_video.write(annotated_frame)
        
        # Save individual frames if requested
        if save_frames:
            frame_path = frames_dir / f'frame_{frame_count:06d}.jpg'
            cv2.imwrite(str(frame_path), annotated_frame)
        
        # Progress indicator
        if frame_count % 30 == 0:
            progress = (frame_count / total_frames) * 100
            print(f"Progress: {progress:.1f}% | Frame {frame_count}/{total_frames} | Faces detected: {num_faces}")
        
        frame_count += 1
    
    # Cleanup
    cap.release()
    if save_annotated_video:
        out_video.release()
    
    # Calculate final statistics
    stats['avg_faces_per_frame'] = stats['total_faces_detected'] / max(stats['total_frames_processed'], 1)
    stats['max_faces_in_frame'] = max(stats['face_count_per_frame']) if stats['face_count_per_frame'] else 0
    
    # Print summary
    print(f"\n{'='*60}")
    print("Processing Complete!")
    print(f"{'='*60}")
    print(f"Frames Processed: {stats['total_frames_processed']}")
    print(f"Total Faces Detected: {stats['total_faces_detected']}")
    print(f"Frames with Faces: {stats['frames_with_faces']}")
    print(f"Average Faces per Frame: {stats['avg_faces_per_frame']:.2f}")
    print(f"Max Faces in Single Frame: {stats['max_faces_in_frame']}")
    print(f"\nOutputs saved to: {output_path}")
    if save_annotated_video:
        print(f"  ✓ Annotated video: {output_video_path}")
    if save_cropped_faces:
        print(f"  ✓ Cropped faces: {faces_dir}/ ({face_id} faces)")
    if save_frames:
        print(f"  ✓ Individual frames: {frames_dir}/")
    print(f"{'='*60}\n")
    
    return stats

print("✓ Video processing function defined!")

# STEP 6: Process Video and Extract Faces

This will:
- Process the video frame by frame
- Detect faces using the trained model
- Save cropped face images
- Create an annotated video with bounding boxes

In [ ]:
# Process the video
stats = extract_faces_from_video(
    video_path=VIDEO_PATH,
    model=model,
    output_dir=OUTPUT_DIR,
    frame_skip=1,              # Process every frame (set to 2 for every other frame, etc.)
    conf_threshold=0.3,        # Confidence threshold (0-1)
    save_annotated_video=True, # Save video with bounding boxes
    save_cropped_faces=True,   # Save individual face crops
    save_frames=False          # Don't save all frames (set True if needed)
)

print("\n✓ Video processing complete!")

# STEP 7: Display Statistics

In [ ]:
print("\n" + "="*60)
print("FACE EXTRACTION STATISTICS")
print("="*60)
print(f"Total Frames Processed: {stats['total_frames_processed']}")
print(f"Total Faces Detected: {stats['total_faces_detected']}")
print(f"Frames with Faces: {stats['frames_with_faces']}")
print(f"Frames without Faces: {stats['total_frames_processed'] - stats['frames_with_faces']}")
print(f"Average Faces per Frame: {stats['avg_faces_per_frame']:.2f}")
print(f"Maximum Faces in a Single Frame: {stats['max_faces_in_frame']}")
print("="*60)

# STEP 8: Display Sample Extracted Faces

Show the first 15 detected and cropped faces

In [ ]:
# Get all detected face images
faces_dir = Path(OUTPUT_DIR) / 'cropped_faces'
face_images = sorted(glob.glob(str(faces_dir / '*.jpg')))[:15]

if len(face_images) == 0:
    print("⚠️ No faces were detected in the video.")
else:
    print(f"\nDisplaying {len(face_images)} sample faces:\n")
    print("="*60)
    
    for i, face_path in enumerate(face_images, 1):
        img = Image.open(face_path)
        print(f"\nFace #{i}: {Path(face_path).name}")
        print(f"Size: {img.size[0]}x{img.size[1]} pixels")
        display(img)
        print("-" * 60)

# STEP 9: List All Output Files

In [ ]:
output_path = Path(OUTPUT_DIR)

print("\n" + "="*60)
print("OUTPUT FILES")
print("="*60)

# Annotated video
video_file = output_path / 'annotated_video.mp4'
if video_file.exists():
    video_size_mb = video_file.stat().st_size / (1024 * 1024)
    print(f"\n📹 Annotated Video:")
    print(f"   Path: {video_file}")
    print(f"   Size: {video_size_mb:.2f} MB")

# Cropped faces
faces_dir = output_path / 'cropped_faces'
if faces_dir.exists():
    face_files = list(faces_dir.glob('*.jpg'))
    total_faces_size_mb = sum(f.stat().st_size for f in face_files) / (1024 * 1024)
    print(f"\n🖼️  Cropped Faces:")
    print(f"   Directory: {faces_dir}")
    print(f"   Total Faces: {len(face_files)}")
    print(f"   Total Size: {total_faces_size_mb:.2f} MB")

# Frames (if saved)
frames_dir = output_path / 'frames'
if frames_dir.exists():
    frame_files = list(frames_dir.glob('*.jpg'))
    if len(frame_files) > 0:
        total_frames_size_mb = sum(f.stat().st_size for f in frame_files) / (1024 * 1024)
        print(f"\n🎞️  Individual Frames:")
        print(f"   Directory: {frames_dir}")
        print(f"   Total Frames: {len(frame_files)}")
        print(f"   Total Size: {total_frames_size_mb:.2f} MB")

print("\n" + "="*60)
print(f"\n✓ All outputs saved in: {output_path}")
print("="*60)

# STEP 10 (Optional): Download Results

If you're running this on Kaggle, the files in `/kaggle/working/` are automatically available for download in the output section on the right side of the screen.

If you want to create a zip file of all extracted faces:

In [ ]:
import shutil

# Create a zip file of all cropped faces
faces_dir = Path(OUTPUT_DIR) / 'cropped_faces'
zip_path = Path(OUTPUT_DIR) / 'extracted_faces'

if faces_dir.exists() and len(list(faces_dir.glob('*.jpg'))) > 0:
    print("Creating zip file of extracted faces...")
    shutil.make_archive(str(zip_path), 'zip', faces_dir)
    
    zip_file = Path(str(zip_path) + '.zip')
    zip_size_mb = zip_file.stat().st_size / (1024 * 1024)
    
    print(f"\n✓ Zip file created!")
    print(f"   Path: {zip_file}")
    print(f"   Size: {zip_size_mb:.2f} MB")
    print(f"\nYou can download this from the Kaggle output panel →")
else:
    print("No faces to zip.")

---

## Summary

This notebook has:
1. ✓ Loaded your pre-trained YOLOv11 face detection model
2. ✓ Processed your video frame by frame
3. ✓ Detected and extracted all faces
4. ✓ Saved individual cropped face images
5. ✓ Created an annotated video with bounding boxes
6. ✓ Provided detailed statistics

### Output Structure:
```
face_extraction_results/
├── annotated_video.mp4           # Video with bounding boxes
├── cropped_faces/                # Individual face images
│   ├── face_000001_0000.jpg
│   ├── face_000001_0001.jpg
│   └── ...
└── extracted_faces.zip           # Zip file of all faces
```

### Next Steps:
- Download the cropped faces for your use case
- Use the annotated video to verify detection quality
- Adjust `conf_threshold` if you need more/fewer detections
- Adjust `frame_skip` to process faster (at the cost of missing some frames)